# Exercise 2 — FastAPI endpoints

**After:** [Day 2](../lessons/day2-fastapi-basics/03_theory_fastapi_fundamentals.md)

In [ ]:
from datetime import date
from enum import Enum

from fastapi import FastAPI, HTTPException, Query, status
from fastapi.testclient import TestClient

def show(r, label=""):
    print(f"{label:<44} {r.status_code}  {r.text[:110]}")

print("ready")

## ⭐ Level 1 — Parameter sources

Given the route `/cities/{name}/weather` and this signature, say where each parameter comes from,
and whether it is required:

```python
def f(name: str, start: date, limit: int = 20, verbose: bool = False)
```

Then write the app and prove it with four requests.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

| Parameter | Source | Required? |
|---|---|---|
| `name` | **path** (it appears in the route) | yes |
| `start` | **query** (scalar, not in the path, no default) | **yes** |
| `limit` | query | no, defaults to 20 |
| `verbose` | query | no, defaults to `False` |

```python
app = FastAPI()

@app.get("/cities/{name}/weather")
def f(name: str, start: date, limit: int = 20, verbose: bool = False):
    return {"name": name, "start": str(start), "limit": limit, "verbose": verbose}

c = TestClient(app)
show(c.get("/cities/Utrecht/weather?start=2026-09-01"), "minimum valid")
show(c.get("/cities/Utrecht/weather"),                  "missing start -> 422")
show(c.get("/cities/Utrecht/weather?start=nope"),       "bad date -> 422")
show(c.get("/cities/Utrecht/weather?start=2026-09-01&limit=5&verbose=yes"), "all options")
```

`start` catches people out: it has no default, so it's **required** despite being a query parameter.
Required-ness comes from the default, not from where the value lives.
</details>

## ⭐⭐ Level 2 — Constrain properly

Write `GET /observations` with:

- `city` required, 1–100 characters
- `sort_by` restricted to `date`, `temp` or `rain` — invalid values must give **422**, and the
  allowed values must show as a **dropdown** in `/docs`
- `limit` between 1 and 500, default 50
- a **422** if `from_date` is after `to_date`

Hint for the dropdown: `Query(pattern=...)` works, but an `Enum` is nicer.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
class SortBy(str, Enum):
    date = "date"
    temp = "temp"
    rain = "rain"

app2 = FastAPI()

@app2.get("/observations")
def observations(
    city: str = Query(min_length=1, max_length=100),
    sort_by: SortBy = SortBy.date,
    limit: int = Query(default=50, ge=1, le=500),
    from_date: date | None = None,
    to_date: date | None = None,
):
    if from_date and to_date and from_date > to_date:
        raise HTTPException(422, detail="from_date must not be after to_date")
    return {"city": city, "sort_by": sort_by, "limit": limit}

c2 = TestClient(app2)
for qs in ["?city=Utrecht", "?city=Utrecht&sort_by=temp", "?city=Utrecht&sort_by=sideways",
           "?city=&limit=5", "?city=Utrecht&limit=9999",
           "?city=Utrecht&from_date=2026-09-10&to_date=2026-09-01"]:
    show(c2.get("/observations" + qs), qs[:42])

print(c2.get("/openapi.json").json()["components"]["schemas"]["SortBy"])
```

`class SortBy(str, Enum)` inherits from `str`, so the member behaves like a string everywhere else in
your code while FastAPI publishes it as an enumerated schema — which is what makes `/docs` render a
dropdown instead of a free-text box.

The date comparison must be a manual `if`: "not after each other" is a relationship between two
fields, and `Query` constraints only see one field at a time. (Day 3's `model_validator(mode="after")`
is the model-level answer to the same problem.)
</details>

## ⭐⭐⭐ Level 3 — Routers and the ordering trap

Split a service into two routers — `health` and `stations` — mount both, and include **three**
station routes that would collide if declared in the wrong order:
`/stations/nearest`, `/stations/summary`, `/stations/{code}`.

Then write a test that proves all three reach the right handler.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
from fastapi import APIRouter

health = APIRouter(tags=["health"])

@health.get("/health")
def h():
    return {"status": "ok"}

stations = APIRouter(prefix="/stations", tags=["stations"])

@stations.get("/nearest")                 # SPECIFIC
def nearest(lat: float, lon: float):
    return {"handler": "nearest", "lat": lat, "lon": lon}

@stations.get("/summary")                 # SPECIFIC
def summary():
    return {"handler": "summary", "count": 2}

@stations.get("/{code}")                  # GENERIC - must be LAST
def read(code: str):
    return {"handler": "read", "code": code}

app3 = FastAPI()
app3.include_router(health)
app3.include_router(stations)

c3 = TestClient(app3)
assert c3.get("/stations/nearest?lat=52&lon=5").json()["handler"] == "nearest"
assert c3.get("/stations/summary").json()["handler"] == "summary"
assert c3.get("/stations/260").json()["handler"] == "read"
assert c3.get("/health").json()["status"] == "ok"
print("all four routed correctly")

for route in app3.routes:
    if hasattr(route, "methods") and not route.path.startswith(("/openapi", "/docs", "/redoc")):
        print("  ", sorted(route.methods), route.path)
```

Now move `@stations.get("/{code}")` to the top and re-run. `/stations/nearest` returns
`{"handler": "read", "code": "nearest"}` — status `200`, plausible JSON, **wrong handler**. That
silent-but-wrong failure mode is exactly why the rule is "specific above generic", and why the
assertions above check the handler name rather than just the status code.
</details>